# 🛡️ Continual Learning Pipeline on Google Colab (GPU)

Môi trường này thực hiện chu trình Học liên tục (Continual Learning - CL) MLOps cho mô hình XLM-R Base phân loại ngôn từ thù ghét. 

### ✨ Chu trình thực thi:
1. **Git Checkout**: Tải mã nguồn nhánh `cl-pipeline` để đồng bộ lịch sử commit.
2. **Mount Drive**: Liên kết Google Drive làm thư mục lưu trữ dữ liệu thô (`02_train_text.csv` và `03_train_label.csv`) và nhận mô hình đầu ra.
3. **Huấn luyện Học liên tục**: Gọi orchestrator `continual_learning.py` để:
   - Chuẩn bị dữ liệu Rehearsal Buffer (lấy mẫu phân tầng 2,500 dòng từ ViHSD Train) & Validation hỗn hợp 50/50.
   - Huấn luyện gia tăng (3 epochs, LR=1e-5, WeightedRandomSampler, Standard Cross-Entropy fallback với Label Smoothing = 0.15).
   - Chạy đánh giá Gatekeeper trên tập Test ViHSD gốc (yêu cầu F1 >= 63.61% để chống Catastrophic Forgetting).
   - Hiệu chuẩn nhiệt độ ECE trên tập validation hỗn hợp.
   - Đóng gói file `CL_output.zip` và tạo tệp đánh dấu `CL_output.zip.success` báo hiệu hoàn tất 100% lên Drive.

### 🚀 Bước 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive vào môi trường Colab
drive.mount('/content/drive')

# Tạo các thư mục cần thiết trên Drive nếu chưa có
os.makedirs("/content/drive/MyDrive/CL_data", exist_ok=True)
os.makedirs("/content/drive/MyDrive/CL_output", exist_ok=True)
print("✅ Mount Google Drive thành công! Vui lòng đảm bảo các file '02_train_text.csv' và '03_train_label.csv' đã ở trong thư mục Drive '/MyDrive/CL_data/'.")

### 📦 Bước 2: Clone repository nhánh `cl-pipeline`

In [ ]:
import os

os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_HOME"] = "/tmp/hf_cache"

REPO_URL = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

# Clone hoặc pull code từ nhánh cl-pipeline
if not os.path.exists(REPO_PATH):
    print(f"📥 Đang clone repository từ nhánh cl-pipeline...")
    !git clone -b cl-pipeline {REPO_URL} {REPO_PATH}
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    !cd {REPO_PATH} && git pull origin cl-pipeline

%cd {REPO_PATH}
print("✅ Đồng bộ mã nguồn nhánh cl-pipeline thành công!")

### 📥 Bước 2.5: Thiết lập Dữ liệu huấn luyện nền tảng (ViHSD Baseline Data)

Vì thư mục dữ liệu `data/` bị bỏ qua bởi git-ignore, ta cần tải và tiền xử lý tập dữ liệu ViHSD gốc để cung cấp mẫu đệm Rehearsal Buffer và thực hiện kiểm tra Gatekeeper.

In [ ]:
# Tải tập dữ liệu ViHSD thô từ GitHub
!python src/data/download.py configs/paths.yaml

# Tiền xử lý dữ liệu sang định dạng .parquet đồng bộ với môi trường local
!python -c "from src.data.preprocessing import process_split; process_split('data/raw/vihsd/train.csv', 'data/processed/train.parquet', use_word_segmentation=False); process_split('data/raw/vihsd/dev.csv', 'data/processed/dev.parquet', use_word_segmentation=False); process_split('data/raw/vihsd/test.csv', 'data/processed/test.parquet', use_word_segmentation=False)"
print("✅ Thiết lập dữ liệu nền tảng ViHSD hoàn tất!")

### 🛠️ Bước 3: Cài đặt thư viện dependencies

In [ ]:
print("📥 Cài đặt các thư viện phục vụ MLOps...")
!pip install -q transformers sentencepiece protobuf huggingface_hub pyyaml pandas pyarrow fastparquet scikit-learn scipy accelerate
print("✅ Cài đặt dependencies hoàn tất!")

### 🚀 Bước 4: Khởi chạy chu trình Học liên tục (Continual Learning)

In [ ]:
# Khởi chạy orchestrator của pipeline học liên tục
# Tệp zip đầu ra sẽ được ghi thẳng vào Google Drive của bạn

!python src/training/continual_learning.py \
    --vlsp-dir "/content/drive/MyDrive/CL_data" \
    --vihsd-train "data/processed/train.parquet" \
    --vihsd-dev "data/processed/dev.parquet" \
    --vihsd-test "data/processed/test.parquet" \
    --output-dir "artifacts/hate_speech_model" \
    --epochs 3 \
    --lr 1e-5 \
    --batch-size 16 \
    --label-smoothing 0.15 \
    --zip-path "/content/drive/MyDrive/CL_output/CL_output.zip"

### 🎉 Hoàn tất!

Sau khi khối lệnh trên chạy thành công:
1. Trên Google Drive tại thư mục `/MyDrive/CL_output/` sẽ xuất hiện 2 tệp tin:
   - `CL_output.zip` (Trọng số và metadata mới đã được đóng gói).
   - `CL_output.zip.success` (Tệp báo hiệu nạp hoàn tất 100%).
2. Google Drive Client trên máy Local sẽ tự động đồng bộ xuống thư mục phục vụ của bạn.
3. Kịch bản `deploy_cl_model.py` cục bộ sẽ tự động phát hiện tệp tin `.success`, thực hiện giải nén an toàn và cập nhật `active_version.json` để hot-reload mô hình tức thời mà không cần restart server!